In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

#assume defauly dir in /255-AI/ dir
os.chdir(Path(globals()['_dh'][0]).parent.parent)
print(os.getcwd())

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

RAW        = Path('data/raw')
PROCESSED  = Path('data/processed')

BLS_FILE  = PROCESSED / 'bls/agg/bls_detail_clean.csv'

ONET_ABILITIES   = RAW / 'onet/Abilities.xlsx'
ONET_ACTIVITIES  = RAW / 'onet/Work Activities.xlsx'
ONET_OCCUPATIONS = RAW / 'onet/Occupation Data.xlsx'

FELTEN_FILE = RAW / 'felten/AIOE_DataAppendix.xlsx'

list(Path('data/').iterdir())
list(Path('data/processed/bls/agg/').iterdir())

xls = pd.ExcelFile(Path('data/raw/felten/AIOE_DataAppendix.xlsx'))
print(xls.sheet_names)

C:\Users\pchau\Documents\MY2S2\CMPE-255\Project\Branch\255-AI
['Index', 'Appendix A', 'Appendix B', 'Appendix C', 'Appendix D', 'Appendix E']


In [2]:
#standardise SOC codes to format XX-XXXX.
def normalize_soc(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r'\.0+$', '', regex=True)
    mask_no_hyphen = ~s.str.contains('-')
    s.loc[mask_no_hyphen] = (
        s.loc[mask_no_hyphen].str.zfill(6)
         .str[:2] + '-' + s.loc[mask_no_hyphen].str.zfill(6).str[2:]
    )
    return s

#lowercase and strip all column names
def col_lower(df):
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    return df

#change to numeric, treating BLS suppression markers ('**', '#') as NaN
def to_numeric_safe(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[#*,]', '', regex=True),
                         errors='coerce')


print('Helpers')

Helpers


In [3]:
#retained columns from detail_clean.csv
BLS_ID_COLS   = ['occ_code', 'occ_title']
BLS_WAGE_COLS = [
    'tot_emp', 'emp_prse',
    'h_mean', 'h_median', 'h_pct10', 'h_pct25', 'h_pct75', 'h_pct90',
    'a_mean', 'a_median', 'a_pct10', 'a_pct25', 'a_pct75', 'a_pct90',
]

#load
bls_raw = col_lower(pd.read_csv(BLS_FILE, dtype=str))
print(f'Raw BLS rows: {len(bls_raw):,} | years present: {sorted(bls_raw["year"].unique())}')

#keep only the columns we need
keep = BLS_ID_COLS + ['year'] + [c for c in BLS_WAGE_COLS if c in bls_raw.columns]
bls_raw = bls_raw[keep].copy()

#normalize SOC key
bls_raw['soc6'] = normalize_soc(bls_raw['occ_code'])

#change wage/employment columns to numeric
for c in [c for c in BLS_WAGE_COLS if c in bls_raw.columns]:
    bls_raw[c] = to_numeric_safe(bls_raw[c])

bls_raw['year'] = bls_raw['year'].astype(int)
print(f'Cleaned BLS rows: {len(bls_raw):,}')
bls_raw.head(3)

Raw BLS rows: 2,450 | years present: ['2019', '2022', '2024']
Cleaned BLS rows: 2,450


,occ_code,occ_title,year,tot_emp,emp_prse,h_mean,h_median,h_pct10,h_pct25,h_pct75,h_pct90,a_mean,a_median,a_pct10,a_pct25,a_pct75,a_pct90,soc6
0,11-1011,Chief Executives,2019,205890,0.8000,93.2000,88.6800,29.9500,54.2300,NaN,NaN,"193,850.0000","184,460.0000","62,290.0000","112,790.0000",NaN,NaN,11-1011
1,11-1021,General and Operations Managers,2019,2400280,0.3000,59.1500,48.4500,21.6600,31.5700,75.6900,NaN,"123,030.0000","100,780.0000","45,050.0000","65,660.0000","157,430.0000",NaN,11-1021
2,11-1031,Legislators,2019,52280,2.2000,NaN,NaN,NaN,NaN,NaN,NaN,"49,440.0000","29,270.0000","17,690.0000","19,070.0000","75,520.0000","100,470.0000",11-1031


In [4]:
#pivot to wide: one row per SOC, wage columns suffixed by year
wage_cols = [c for c in bls_raw.columns if c in BLS_WAGE_COLS]

bls_wide_frames = []
for yr, grp in bls_raw.groupby('year'):
    renamed = {c: f'{c}_{yr}' for c in wage_cols}
    tmp = grp.rename(columns=renamed).drop(columns='year')
    tmp = tmp.set_index('soc6')
    bls_wide_frames.append(tmp)

#outer-join vintages so all SOC codes from any year are retained
bls = bls_wide_frames[0]
for frame in bls_wide_frames[1:]:
    bls = bls.join(frame, how='outer', rsuffix='_dup')

#resolve duplicate occ_title columns, keep the most recent non-null value
title_cols = [c for c in bls.columns if c.startswith('occ_title')]
bls['occ_title'] = bls[title_cols].bfill(axis=1).iloc[:, 0]
bls = bls.drop(columns=[c for c in title_cols if c != 'occ_title'])
bls = bls.drop(columns=[c for c in bls.columns if '_dup' in c])

bls = bls.reset_index()

#move SOC code and title to front
front = ['soc6', 'occ_title']
bls = bls[front + [c for c in bls.columns if c not in front]]

print(f'BLS wide: {bls["soc6"].nunique():,} unique SOC codes | {bls.shape[1]} columns')
bls.head(3)

BLS wide: 861 unique SOC codes | 45 columns


,soc6,occ_title,occ_code,tot_emp_2019,emp_prse_2019,h_mean_2019,h_median_2019,h_pct10_2019,h_pct25_2019,h_pct75_2019,h_pct90_2019,a_mean_2019,a_median_2019,a_pct10_2019,a_pct25_2019,a_pct75_2019,a_pct90_2019,tot_emp_2022,emp_prse_2022,h_mean_2022,h_median_2022,h_pct10_2022,h_pct25_2022,h_pct75_2022,h_pct90_2022,a_mean_2022,a_median_2022,a_pct10_2022,a_pct25_2022,a_pct75_2022,a_pct90_2022,tot_emp_2024,emp_prse_2024,h_mean_2024,h_median_2024,h_pct10_2024,h_pct25_2024,h_pct75_2024,h_pct90_2024,a_mean_2024,a_median_2024,a_pct10_2024,a_pct25_2024,a_pct75_2024,a_pct90_2024
0,11-1011,Chief Executives,11-1011,"205,890.0000",0.8000,93.2000,88.6800,29.9500,54.2300,NaN,NaN,"193,850.0000","184,460.0000","62,290.0000","112,790.0000",NaN,NaN,"199,240.0000",0.9000,118.4800,91.1200,36.0200,58.8900,NaN,NaN,"246,440.0000","189,520.0000","74,920.0000","122,480.0000",NaN,NaN,"211,850.0000",1.2000,126.4100,99.2400,35.4400,60.6100,NaN,NaN,"262,930.0000","206,420.0000","73,710.0000","126,080.0000",NaN,NaN
1,11-1021,General and Operations Managers,11-1021,"2,400,280.0000",0.3000,59.1500,48.4500,21.6600,31.5700,75.6900,NaN,"123,030.0000","100,780.0000","45,050.0000","65,660.0000","157,430.0000",NaN,"3,376,680.0000",0.3000,59.0700,47.1600,20.9000,29.8400,74.3100,106.3800,"122,860.0000","98,100.0000","43,470.0000","62,070.0000","154,560.0000","221,270.0000","3,584,420.0000",0.4000,64.0000,49.5000,22.8000,32.2900,78.9100,NaN,"133,120.0000","102,950.0000","47,420.0000","67,160.0000","164,130.0000",NaN
2,11-1031,Legislators,11-1031,"52,280.0000",2.2000,NaN,NaN,NaN,NaN,NaN,NaN,"49,440.0000","29,270.0000","17,690.0000","19,070.0000","75,520.0000","100,470.0000","42,890.0000",2.1000,NaN,NaN,NaN,NaN,NaN,NaN,"71,100.0000","48,090.0000","20,970.0000","28,690.0000","94,030.0000","149,710.0000","26,510.0000",3.9000,NaN,NaN,NaN,NaN,NaN,NaN,"67,390.0000","44,810.0000","20,380.0000","29,120.0000","80,350.0000","137,820.0000"


In [5]:
#ONET occupation data
onet_occ = col_lower(pd.read_excel(ONET_OCCUPATIONS, dtype=str))

#find the SOC column
soc_col = next(c for c in onet_occ.columns if 'soc' in c or 'code' in c)
onet_occ = onet_occ.rename(columns={soc_col: 'onet_soc'})

#O*NET trim to 6-digit XX-XXXX
onet_occ['soc6'] = normalize_soc(onet_occ['onet_soc'].str[:7])

#keep title and description columns only
keep_cols = [c for c in onet_occ.columns if c not in ('onet_soc',) and
             ('title' in c or 'description' in c)]
onet_occ = onet_occ[['soc6'] + keep_cols].copy()

#prefix with onet_ for namespacing
onet_occ.columns = ['soc6'] + [f'onet_{c}' for c in onet_occ.columns if c != 'soc6']

#if multiple O*NET sub-occupations map to the same 6-digit SOC, keep the first
onet_occ = onet_occ.drop_duplicates(subset='soc6', keep='first').reset_index(drop=True)

print(f'O*NET Occupation Data: {len(onet_occ):,} SOC codes | columns: {list(onet_occ.columns)}')

O*NET Occupation Data: 867 SOC codes | columns: ['soc6', 'onet_title', 'onet_description']


In [6]:
#Load ONET element, pivot long to wide
def pivot_onet_ratings(path, label):
    df = col_lower(pd.read_excel(path, dtype=str))

    #identify key columns flexibly
    soc_col   = next(c for c in df.columns if 'soc' in c or 'code' in c)
    elem_col  = next((c for c in df.columns if 'element_name' in c), None)
    scale_col = next((c for c in df.columns if 'scale_id' in c), None)
    data_col  = next((c for c in df.columns if 'data_value' in c), None)

    if any(x is None for x in [elem_col, scale_col, data_col]):
        raise ValueError(f'Cannot identify required columns in {path}. Found: {list(df.columns)}')

    df['soc6'] = normalize_soc(df[soc_col].str[:7])
    df[data_col] = pd.to_numeric(df[data_col], errors='coerce')

    #prefer Importance (IM) scale or fall back to Level (LV)
    scales = df[scale_col].unique()
    scale_pref = 'IM' if 'IM' in scales else ('LV' if 'LV' in scales else scales[0])
    sub = df[df[scale_col] == scale_pref].copy()

    #average across multiple raters for the same SOC and element
    sub = sub.groupby(['soc6', elem_col])[data_col].mean().reset_index()

    #pivot to wide
    wide = sub.pivot(index='soc6', columns=elem_col, values=data_col)
    wide.columns = [
        f'{label}__{re.sub(r"[^a-z0-9]+", "_", c.lower().strip())}'
        for c in wide.columns
    ]
    wide = wide.reset_index()

    print(f'O*NET {label}: {len(wide):,} SOC codes | {wide.shape[1]-1} elements')
    return wide


onet_abilities  = pivot_onet_ratings(ONET_ABILITIES,  'ability')
onet_activities = pivot_onet_ratings(ONET_ACTIVITIES, 'workact')

#merge all O*NET tables
onet = onet_occ.merge(onet_abilities,  on='soc6', how='inner')
onet = onet.merge(onet_activities, on='soc6', how='inner')

print(f'O*NET merged: {len(onet):,} SOC codes | {onet.shape[1]} columns')
onet.head(3)

O*NET ability: 774 SOC codes | 52 elements
O*NET workact: 774 SOC codes | 41 elements
O*NET merged: 774 SOC codes | 96 columns


,soc6,onet_title,onet_description,ability__arm_hand_steadiness,ability__auditory_attention,ability__category_flexibility,ability__control_precision,ability__deductive_reasoning,ability__depth_perception,ability__dynamic_flexibility,ability__dynamic_strength,ability__explosive_strength,ability__extent_flexibility,ability__far_vision,ability__finger_dexterity,ability__flexibility_of_closure,ability__fluency_of_ideas,ability__glare_sensitivity,ability__gross_body_coordination,ability__gross_body_equilibrium,ability__hearing_sensitivity,ability__inductive_reasoning,ability__information_ordering,ability__manual_dexterity,ability__mathematical_reasoning,ability__memorization,ability__multilimb_coordination,ability__near_vision,ability__night_vision,ability__number_facility,...,workact__establishing_and_maintaining_interpersonal_relationships,workact__estimating_the_quantifiable_characteristics_of_products_events_or_information,workact__evaluating_information_to_determine_compliance_with_standards,workact__getting_information,workact__guiding_directing_and_motivating_subordinates,workact__handling_and_moving_objects,workact__identifying_objects_actions_and_events,workact__inspecting_equipment_structures_or_materials,workact__interpreting_the_meaning_of_information_for_others,workact__judging_the_qualities_of_objects_services_or_people,workact__making_decisions_and_solving_problems,workact__monitoring_processes_materials_or_surroundings,workact__monitoring_and_controlling_resources,workact__operating_vehicles_mechanized_devices_or_equipment,workact__organizing_planning_and_prioritizing_work,workact__performing_administrative_activities,workact__performing_general_physical_activities,workact__performing_for_or_working_directly_with_the_public,workact__processing_information,workact__providing_consultation_and_advice_to_others,workact__repairing_and_maintaining_electronic_equipment,workact__repairing_and_maintaining_mechanical_equipment,workact__resolving_conflicts_and_negotiating_with_others,workact__scheduling_work_and_activities,workact__selling_or_influencing_others,workact__staffing_organizational_units,workact__thinking_creatively,workact__training_and_teaching_others,workact__updating_and_using_relevant_knowledge,workact__working_with_computers
0,11-1011,Chief Executives,Determine and formulate policies and provide o...,1.1900,2.0600,3.3100,1.6250,4.0000,1.8750,1.0000,1.1250,1.0000,1.0000,2.9400,1.6250,3.0650,3.8800,1.1250,1.0000,1.0000,2.0600,3.9400,3.8100,1.0000,3.0000,2.6300,1.6250,3.3700,1.1250,3.0000,...,4.7350,3.4000,4.0800,4.6700,4.2600,1.6500,4.0400,2.3100,4.2950,4.0150,4.7400,3.8650,3.9950,2.1850,4.4500,3.6600,1.8850,3.7300,4.2100,3.9400,1.4650,1.3100,3.9550,3.7300,3.9600,3.5200,4.3250,3.8200,4.3050,4.1600
1,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",1.6200,2.1200,3.3800,1.1200,3.8800,1.8800,1.0000,1.1200,1.3800,1.2500,2.6200,1.5000,2.6200,3.2500,1.0000,1.5000,1.3800,1.7500,3.5000,3.5000,1.5000,2.8800,2.2500,1.5000,3.5000,1.1200,2.8800,...,4.2800,3.2300,3.5900,4.4200,4.2000,2.0000,4.2200,2.6200,3.7200,4.1000,4.3300,3.9100,3.8600,1.9000,4.2000,3.4000,2.1100,2.5400,4.0800,3.6600,1.8900,1.8300,3.8800,3.8100,3.5800,3.7000,3.7000,3.4100,3.8000,4.4600
2,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",1.3800,1.7500,3.3800,1.1200,3.8800,1.7500,1.0000,1.0000,1.0000,1.0000,2.8800,1.6200,3.1200,3.7500,1.0000,1.0000,1.0000,1.7500,3.5000,3.2500,1.3800,3.0000,2.5000,1.0000,3.7500,1.0000,2.8800,...,4.0100,2.7900,2.3500,4.3200,2.9800,2.0800,3.6000,1.7700,3.0100,3.3700,3.8200,2.7800,2.5200,1.8200,4.1700,3.1100,1.9000,3.1800,3.5100,2.3100,1.3400,1.2100,2.8200,3.3400,3.6400,2.2900,4.0500,2.7400,3.7800,4.6100


In [7]:
#inspect sheets
xls = pd.ExcelFile(FELTEN_FILE)
print('Sheets found:', xls.sheet_names)

#load appendix A
felten_raw = col_lower(xls.parse('Appendix A', dtype=str))
print('Appendix A columns:', felten_raw.columns.tolist())

#find SOC column
soc_col = next(
    (c for c in felten_raw.columns if 'soc' in c or 'code' in c),
    None
)
if soc_col is None:
    raise ValueError(f'Cannot find SOC column. Columns: {list(felten_raw.columns)}')

felten_raw['soc6'] = normalize_soc(felten_raw[soc_col])

#score columns: everything that isn't an identifier or text field
score_cols = [
    c for c in felten_raw.columns
    if c not in (soc_col, 'soc6')
    and not any(kw in c for kw in ('title', 'name', 'description', 'occupation'))
]

for c in score_cols:
    felten_raw[c] = pd.to_numeric(felten_raw[c], errors='coerce')

#prefix with aioe_ for spacing
rename_map = {c: f'aioe_{c}' for c in score_cols}
felten = felten_raw.rename(columns=rename_map)
felten = felten[['soc6'] + list(rename_map.values())]

#average if duplicate SOC codes exist
felten = felten.groupby('soc6').mean(numeric_only=True).reset_index()

print(f'Felten AIOE: {len(felten):,} SOC codes | {felten.shape[1]-1} score columns')
felten.head(3)



Sheets found: ['Index', 'Appendix A', 'Appendix B', 'Appendix C', 'Appendix D', 'Appendix E']
Appendix A columns: ['soc_code', 'occupation_title', 'aioe']
Felten AIOE: 774 SOC codes | 1 score columns


,soc6,aioe_aioe
0,11-1011,1.3342
1,11-1021,0.5749
2,11-2011,1.2944


In [8]:
#coverage before merge
print('Pre-merge SOC coverage')
print(f'  BLS (any vintage) : {bls["soc6"].nunique():>5,}')
print(f'  O*NET             : {onet["soc6"].nunique():>5,}')
print(f'  Felten AIOE       : {felten["soc6"].nunique():>5,}')

#triple inner join on SOC code
master = (
    bls
    .merge(onet,   on='soc6', how='inner', suffixes=('', '_onet'))
    .merge(felten, on='soc6', how='inner', suffixes=('', '_felten'))
)

#move identifiers to front
id_cols    = ['soc6', 'occ_title']
other_cols = [c for c in master.columns if c not in id_cols]
master = master[id_cols + other_cols]

print(f'Master table: {len(master):,} occupations | {master.shape[1]:,} columns')
print(f'  BLS columns   : {sum(1 for c in master.columns if any(str(y) in c for y in [2019,2022,2024]))}')
print(f'  O*NET columns : {sum(1 for c in master.columns if c.startswith(("onet_","ability__","workact__")))}')
print(f'  Felten columns: {sum(1 for c in master.columns if c.startswith("aioe_"))}')

Pre-merge SOC coverage
  BLS (any vintage) :   861
  O*NET             :   774
  Felten AIOE       :   774
Master table: 668 occupations | 141 columns
  BLS columns   : 42
  O*NET columns : 95
  Felten columns: 1


In [9]:
#duplicate SOC codes
dupe_mask = master['soc6'].duplicated(keep=False)
n_dupes = dupe_mask.sum()
if n_dupes > 0:
    print(f'WARNING: {n_dupes} duplicate soc6 values — inspect below:')
    display(master[dupe_mask][['soc6', 'occ_title']].sort_values('soc6'))
else:
    print('No duplicate SOC codes.')

#missingness
miss = (master.isnull().mean() * 100).round(2).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f'Columns with any missing data: {len(miss_nonzero)}')
miss_nonzero.head(20)

#coverage diagram
bls_set    = set(bls['soc6'])
onet_set   = set(onet['soc6'])
felten_set = set(felten['soc6'])
master_set = set(master['soc6'])

print('SOC code set sizes')
print(f'  BLS           : {len(bls_set):,}')
print(f'  O*NET         : {len(onet_set):,}')
print(f'  Felten        : {len(felten_set):,}')
print(f'  BLS ∩ O*NET   : {len(bls_set & onet_set):,}')
print(f'  BLS ∩ Felten  : {len(bls_set & felten_set):,}')
print(f'  O*NET ∩ Felten: {len(onet_set & felten_set):,}')
print(f'  Triple ∩ (=Master): {len(master_set):,}')

#SOC codes dropped by each join
not_in_onet   = bls_set - onet_set
not_in_felten = (bls_set & onet_set) - felten_set

print(f'BLS occupations missing from O*NET   : {len(not_in_onet):,}')
print(f'BLS∩O*NET occupations missing from Felten: {len(not_in_felten):,}')

if not_in_onet:
    print('\nSample SOC codes in BLS but not O*NET:', sorted(not_in_onet)[:10])
if not_in_felten:
    print('\nSample SOC codes in BLS∩O*NET but not Felten:', sorted(not_in_felten)[:10])

#check key wage fields
wage_check_cols = [c for c in master.columns if 'a_mean' in c or 'a_median' in c]
display(master[['soc6','occ_title'] + wage_check_cols].describe())

No duplicate SOC codes.
Columns with any missing data: 41
SOC code set sizes
  BLS           : 861
  O*NET         : 774
  Felten        : 774
  BLS ∩ O*NET   : 750
  BLS ∩ Felten  : 674
  O*NET ∩ Felten: 682
  Triple ∩ (=Master): 668
BLS occupations missing from O*NET   : 111
BLS∩O*NET occupations missing from Felten: 82

Sample SOC codes in BLS but not O*NET: ['11-1031', '11-2030', '11-2032', '11-3010', '11-9039', '11-9198', '13-1020', '13-1082', '13-1198', '13-2020']

Sample SOC codes in BLS∩O*NET but not Felten: ['11-2033', '11-3012', '11-3013', '11-9072', '11-9171', '11-9179', '15-1211', '15-1212', '15-1221', '15-1231']


,a_mean_2019,a_median_2019,a_mean_2022,a_median_2022,a_mean_2024,a_median_2024
count,658.0000,655.0000,664.0000,663.0000,665.0000,662.0000
mean,"61,109.8024","55,726.3053","67,609.0211","61,150.1508","73,364.0902","66,261.9184"
std,"30,779.6356","26,104.8761","34,783.1526","28,131.9425","37,022.6875","29,509.6784"
min,"22,910.0000","21,260.0000","27,870.0000","27,270.0000","30,830.0000","30,160.0000"
25%,"38,835.0000","36,300.0000","43,870.0000","40,010.0000","48,760.0000","46,052.5000"
50%,"52,795.0000","48,980.0000","57,950.0000","51,570.0000","63,430.0000","58,640.0000"
75%,"74,195.0000","68,690.0000","83,172.5000","75,605.0000","87,600.0000","78,922.5000"
max,"237,570.0000","184,460.0000","358,080.0000","211,790.0000","360,240.0000","226,600.0000"


In [10]:
OUT_EXCEL  = PROCESSED / 'master_occupation_table.xlsx'
OUT_PARQUET = PROCESSED / 'master_occupation_table.parquet'

#excel (two sheets: full table + data dict)
with pd.ExcelWriter(OUT_EXCEL, engine='openpyxl') as writer:
    master.to_excel(writer, sheet_name='master', index=False)

    #data dict
    def col_source(col):
        if col in ('soc6', 'occ_title'):        return 'BLS'
        if any(str(y) in col for y in [2019,2022,2024]): return 'BLS OEWS'
        if col.startswith(('ability__','workact__')): return 'O*NET 30.1'
        if col.startswith('onet_'):             return 'O*NET 30.1'
        if col.startswith('aioe_'):             return 'Felten AIOE main'
        if col.startswith('genai_'):            return 'Felten AIOE GenAI'
        return 'unknown'

    dd = pd.DataFrame({
        'column'      : master.columns,
        'source'      : [col_source(c) for c in master.columns],
        'dtype'       : [str(master[c].dtype) for c in master.columns],
        'pct_missing' : [round(master[c].isnull().mean()*100, 2) for c in master.columns],
        'sample_value': [master[c].dropna().iloc[0] if master[c].notna().any() else '' for c in master.columns],
    })
    dd.to_excel(writer, sheet_name='data_dictionary', index=False)

#parquet (could be used for modeling)
master.to_parquet(OUT_PARQUET, index=False)

print(f'Saved Excel  → {OUT_EXCEL}')
print(f'Saved Parquet → {OUT_PARQUET}')
print(f'\nFinal dimensions: {master.shape[0]:,} rows × {master.shape[1]:,} columns')
master[['soc6','occ_title']].head(10)

Saved Excel  → data\processed\master_occupation_table.xlsx
Saved Parquet → data\processed\master_occupation_table.parquet

Final dimensions: 668 rows × 141 columns


,soc6,occ_title
0,11-1011,Chief Executives
1,11-1021,General and Operations Managers
2,11-2011,Advertising and Promotions Managers
3,11-2021,Marketing Managers
4,11-2022,Sales Managers
5,11-3021,Computer and Information Systems Managers
6,11-3031,Financial Managers
7,11-3051,Industrial Production Managers
8,11-3061,Purchasing Managers
9,11-3071,"Transportation, Storage, and Distribution Mana..."
